# Practice: text as numbers, tokens
*AI from scratch, stage 5* · lesson: https://ihor.sh/ai/tokens.html

Count UTF-8 bytes, build byte-pair encoding from scratch on the NYC station names (pair counts, merges, training,
encoding new text), then open GPT-2's real tokenizer and measure what the lesson claims about "strawberry" and
about English versus Ukrainian. Replace each `...` under `# YOUR CODE HERE`, run the cell, then the check under it;
`hint(n)` and `answer(n)` are there when you're stuck.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['len("Привіт".encode("utf-8"))'], 2: ['Counter((a, b) for w in words for a, b in zip(w, w[1:]))'], 3: ['A while loop over i: if i < len(w) - 1 and w[i] == a and w[i + 1] == b, append a + b and move i by 2.', 'Otherwise append w[i] and move i by 1. Return out at the end.'], 4: ['for step in range(200):   then a, b = count_pairs(words).most_common(1)[0][0]', 'merges.append((a, b)); words = [merge(w, a, b) for w in words]; avg_tokens.append(sum(map(len, words)) / len(words))'], 5: ['w = list(text), then for a, b in merges: w = merge(w, a, b). Return w.'], 6: ['en = len(tok.encode(english)); uk = len(tok.encode(ukrainian))', 'ratio = uk / en'], 7: ['pieces = [tok.decode([i]) for i in tok.encode("strawberry")]', 'the same with " strawberry" for pieces_space']}
_ANSWERS = {1: 'bl9ieXRlcyA9IGxlbigi0J/RgNC40LLRltGCIi5lbmNvZGUoInV0Zi04Iikp', 2: 'cmV0dXJuIENvdW50ZXIoKGEsIGIpIGZvciB3IGluIHdvcmRzIGZvciBhLCBiIGluIHppcCh3LCB3WzE6XSkp', 3: 'd2hpbGUgaSA8IGxlbih3KToKICAgIGlmIGkgPCBsZW4odykgLSAxIGFuZCB3W2ldID09IGEgYW5kIHdbaSArIDFdID09IGI6CiAgICAgICAgb3V0LmFwcGVuZChhICsgYik7IGkgKz0gMgogICAgZWxzZToKICAgICAgICBvdXQuYXBwZW5kKHdbaV0pOyBpICs9IDEKcmV0dXJuIG91dA==', 4: 'Zm9yIHN0ZXAgaW4gcmFuZ2UoMjAwKToKICAgIGEsIGIgPSBjb3VudF9wYWlycyh3b3JkcykubW9zdF9jb21tb24oMSlbMF1bMF0KICAgIG1lcmdlcy5hcHBlbmQoKGEsIGIpKQogICAgd29yZHMgPSBbbWVyZ2UodywgYSwgYikgZm9yIHcgaW4gd29yZHNdCiAgICBhdmdfdG9rZW5zLmFwcGVuZChzdW0obWFwKGxlbiwgd29yZHMpKSAvIGxlbih3b3Jkcykp', 5: 'dyA9IGxpc3QodGV4dCkKZm9yIGEsIGIgaW4gbWVyZ2VzOgogICAgdyA9IG1lcmdlKHcsIGEsIGIpCnJldHVybiB3', 6: 'ZW4gPSBsZW4odG9rLmVuY29kZShlbmdsaXNoKSkKdWsgPSBsZW4odG9rLmVuY29kZSh1a3JhaW5pYW4pKQpyYXRpbyA9IHVrIC8gZW4=', 7: 'cGllY2VzID0gW3Rvay5kZWNvZGUoW2ldKSBmb3IgaSBpbiB0b2suZW5jb2RlKCJzdHJhd2JlcnJ5IildCnBpZWNlc19zcGFjZSA9IFt0b2suZGVjb2RlKFtpXSkgZm9yIGkgaW4gdG9rLmVuY29kZSgiIHN0cmF3YmVycnkiKV0='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import urllib.request
from collections import Counter
import matplotlib.pyplot as plt

text = urllib.request.urlopen("https://raw.githubusercontent.com/ikruchynskyi/ihor.sh/main/ai/data/stations.txt").read().decode()
names = [n for n in text.split("\n") if n]
print(len(names), "names, for example:", names[:5])

## 1. Bytes
Set `n_bytes` to the number of UTF-8 bytes in `"Привіт"` ("hello" in Ukrainian): `.encode("utf-8")` gives the bytes.

In [ ]:
n_bytes = None
# YOUR CODE HERE (stuck? hint(1), then answer(1))
...
print(n_bytes)

In [ ]:
check(1, lambda: n_bytes == 12, "6 Cyrillic letters × 2 bytes each = 12")

## 2. Count neighboring pairs
Write `count_pairs(words)`: `words` is a list of token lists (like `[["C", "o", "u"], ["A", "v"]]`). Return a
`Counter` of every neighboring pair `(a, b)` over all the words, as the lesson's box does with `zip(w, w[1:])`.

In [ ]:
def count_pairs(words):
    # YOUR CODE HERE (stuck? hint(2), then answer(2))
    ...

In [ ]:
check(2, lambda: count_pairs([["a", "b", "a", "b"], ["a", "b"]]) == Counter({("a", "b"): 3, ("b", "a"): 1}),
      "('a','b') appears 3 times (twice in the first word, once in the second), ('b','a') once")

## 3. Merge one pair
Write `merge(w, a, b)`: walk through the token list `w`; wherever `a` is directly followed by `b`, put the single
token `a + b`, otherwise keep the token. (The lesson's box has one; write it without looking if you can.)

In [ ]:
def merge(w, a, b):
    out, i = [], 0
    # YOUR CODE HERE (stuck? hint(3), then answer(3))
    ...

In [ ]:
check(3, lambda: merge(["S", "t", " ", "S", "t", "S"], "S", "t") == ["St", " ", "St", "S"] and merge(["a", "a", "a"], "a", "a") == ["aa", "a"],
      "pairs merge left to right and don't overlap: 'aaa' becomes 'aa', 'a'")

## 4. Train BPE
Starting from `words` (every name as characters), do 200 merges: each time, find the most common pair with
`count_pairs(words).most_common(1)[0][0]`, append it to `merges`, and replace `words` by every word merged.
`avg_tokens` (given) then records the average tokens per name after each merge.

In [ ]:
words = [list(n) for n in names]
merges, avg_tokens = [], [sum(map(len, words)) / len(words)]
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print("first 10 merges:", [a + b for a, b in merges[:10]])
plt.plot(avg_tokens); plt.xlabel("merges"); plt.ylabel("average tokens per station name"); plt.show()

In [ ]:
check(4, lambda: len(merges) == 200 and merges[0] == (" ", "S") and 5.4 < avg_tokens[-1] < 6.0,
      "the first merge is ' ' + 'S' (all those ' St' and ' Sq'); after 200 merges a name is about 5.7 tokens instead of 12")

## 5. Tokenize new text
Write `encode(text)`: start from `list(text)` and apply every learned merge, in the order learned.

In [ ]:
def encode(text):
    # YOUR CODE HERE (stuck? hint(5), then answer(5))
    ...

for t in ["Court Sq", "Times Sq-42 St", "Strawberry Av", "Kyiv Av"]:
    print(repr(t), "→", encode(t))

In [ ]:
check(5, lambda: encode("Court Sq")[-1] == " Sq" and "".join(encode("Zzyzx Rd")) == "Zzyzx Rd",
      "learned pieces are used, and a never-seen word still comes out whole, just in smaller pieces")

## 6. A real tokenizer: GPT-2's
Hugging Face's `transformers` (preinstalled on Colab and Kaggle) downloads GPT-2's tokenizer, 50,257 tokens learned
with byte-level BPE. Set `en` and `uk` to the number of tokens (`len(tok.encode(...))`) for the English and Ukrainian
sentences, and `ratio = uk / en`.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("gpt2")
english, ukrainian = "Hello, how are you today?", "Привіт, як ти сьогодні?"
en = uk = ratio = None
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...
print(en, uk, ratio)

In [ ]:
check(6, lambda: (en, uk) == (7, 26) and abs(ratio - 26 / 7) < 1e-9, "7 against 26: the same question costs 3.7 times as many tokens in Ukrainian")

## 7. What does the model see of "strawberry"?
Set `pieces` to the list of strings for each token of `"strawberry"`: encode it, then decode each id on its own
(`tok.decode([i])`). Then set `pieces_space` the same way for `" strawberry"` (with a leading space).

In [ ]:
pieces = pieces_space = None
# YOUR CODE HERE (stuck? hint(7), then answer(7))
...
print(pieces, pieces_space, tok.encode(" strawberry"))

In [ ]:
check(7, lambda: pieces == ["st", "raw", "berry"] and pieces_space == [" strawberry"],
      "three pieces at the start of a line, one id with a space: either way, no letters for the model to count")

## Next
Back to the path: https://ihor.sh/ai/ · Karpathy, "Let's build the GPT Tokenizer": https://www.youtube.com/watch?v=zduSFxRajkE